# LangChain: Reranking با Cross-Encoder

## Outline
* مشکل Bi-Encoder در Retrieval
* مفهوم Cross-Encoder و Reranking
* استفاده مستقیم از `BAAI/bge-reranker-v2-m3`
* ادغام Reranker در LangChain با `ContextualCompressionRetriever`
* RAG کامل با Reranker (Retrieve → Rerank → Generate)
* مقایسه Bi-Encoder و Cross-Encoder


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())


## ۱. مشکل Bi-Encoder و راه‌حل Reranking

<div dir="rtl">
در جلسات قبل، برای Retrieval از <b>Embedding + Similarity Search</b> استفاده کردیم. به این روش <b>Bi-Encoder</b> می‌گویند، چون query و document <b>جداگانه</b> به بردار تبدیل می‌شوند و بعد فاصله‌ی آن‌ها مقایسه می‌شود:

</div>

```
Query    → [Encoder] → vector ↘
                                cosine similarity → score
Document → [Encoder] → vector ↗
```

<div dir="rtl">
<b>مزیت:</b> بسیار سریع است — بردار همه‌ی داکیومنت‌ها از قبل محاسبه و در Vector Store ذخیره شده است.
</div>

<div dir="rtl">
<b>مشکل:</b> چون query و document هرگز «هم‌دیگر را نمی‌بینند»، مدل نمی‌تواند تعامل دقیق کلمات را بفهمد. مثلاً داکیومنتی که فقط کلمات مشابه دارد (اما جواب سؤال نیست) ممکن است امتیاز بالایی بگیرد.
</div>

<div dir="rtl">
<b>راه‌حل: Cross-Encoder.</b> در این روش query و document <b>با هم</b> به مدل داده می‌شوند و مدل یک امتیاز ارتباط (relevance score) مستقیم تولید می‌کند:

</div>

```
[Query + Document] → [Cross-Encoder] → relevance score
```

<div dir="rtl">
Cross-Encoder بسیار دقیق‌تر است، اما کند — چون برای هر جفت (query, document) باید یک بار کل مدل اجرا شود و نمی‌توان چیزی را از قبل محاسبه کرد. پس راه‌حل عملی، یک pipeline <b>دو مرحله‌ای</b> است:

</div>

```
User Query → Retriever (Bi-Encoder, k=10)  ← سریع، بازیابی گسترده
           → Reranker  (Cross-Encoder)      ← دقیق، مرتب‌سازی مجدد
           → Top 3 Docs → LLM → Answer
```

<div dir="rtl">
یعنی: اول  Bi-Encoder تعداد زیادی کاندید می‌آوریم (سریع)، بعد با Cross-Encoder فقط همین کاندیدها را دقیق امتیازدهی و مرتب می‌کنیم.

</div>


## ۲. ساخت Knowledge Base

<div dir="rtl">
مثل قبل یک کاتالوگ فروشگاهی می‌سازیم — اما این بار چند داکیومنت «گمراه‌کننده» هم اضافه می‌کنیم (مثل کرم ضدآفتاب و عینک آفتابی) تا ضعف Bi-Encoder را ببینیم.

</div>


In [2]:
# pip install langchain-openai langchain-community faiss-cpu

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

docs = [
    Document(page_content="پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را مسدود می‌کند. قابل شستشو با ماشین لباسشویی.", metadata={"source": "catalog", "id": 1}),
    Document(page_content="مایوهای شنای ما بسیار راحت هستند و سریع خشک می‌شوند. موجود در ۶ رنگ.", metadata={"source": "catalog", "id": 2}),
    Document(page_content="چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده‌روی ارائه می‌دهند.", metadata={"source": "catalog", "id": 3}),
    Document(page_content="کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.", metadata={"source": "catalog", "id": 4}),
    Document(page_content="ژاکت پشمی در هوای سرد گرما را حفظ می‌کند. قابل شستشو با ماشین لباسشویی و با قابلیت بسته‌بندی آسان.", metadata={"source": "catalog", "id": 5}),
    Document(page_content="کرم ضدآفتاب SPF 50 پوست را در برابر آفتاب سوختگی محافظت می‌کند. مقاوم در برابر آب و تعریق.", metadata={"source": "catalog", "id": 6}),
    Document(page_content="عینک آفتابی پلاریزه با لنز UV400، چشم را در برابر نور شدید خورشید محافظت می‌کند.", metadata={"source": "catalog", "id": 7}),
    Document(page_content="شلوار کوهنوردی سبک و سریع خشک‌شونده، مناسب مسیرهای طولانی در تابستان.", metadata={"source": "catalog", "id": 8}),
    Document(page_content="دستکش زمستانی ضدآب با لایه داخلی گرم، مناسب اسکی و کوهنوردی در برف.", metadata={"source": "catalog", "id": 9}),
    Document(page_content="کیف ضدآب مخصوص قایق‌سواری، وسایل شما را کاملاً خشک نگه می‌دارد.", metadata={"source": "catalog", "id": 10}),
]
print(f"Loaded {len(docs)} documents")


Loaded 10 documents


C:\Users\Alireza\AppData\Local\Temp\ipykernel_53828\1910527394.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [3]:
# ساخت vector store (Bi-Encoder)
embeddings = OllamaEmbeddings(model="bge-m3")
vectorstore = FAISS.from_documents(docs, embeddings)
print("Vector store created!")

Vector store created!


In [4]:
# ── مرحله ۱: Retrieval با Bi-Encoder ──
query = "چه لباسی پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت می‌کند؟"

bi_results = vectorstore.similarity_search_with_relevance_scores(query, k=6)

print(f"Bi-Encoder ranking for: '{query}'\n")
for i, (doc, score) in enumerate(bi_results):
    print(f"  {i+1}. [{score*100:.1f}%] {doc.page_content[:70]}")


Bi-Encoder ranking for: 'چه لباسی پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت می‌کند؟'

  1. [39.1%] پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های 
  2. [34.9%] کرم ضدآفتاب SPF 50 پوست را در برابر آفتاب سوختگی محافظت می‌کند. مقاوم 
  3. [34.2%] چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده
  4. [34.2%] کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طر
  5. [30.1%] عینک آفتابی پلاریزه با لنز UV400، چشم را در برابر نور شدید خورشید محاف
  6. [26.1%] شلوار کوهنوردی سبک و سریع خشک‌شونده، مناسب مسیرهای طولانی در تابستان.


<div dir="rtl">
به رتبه‌بندی بالا دقت کنید: احتمالاً <b>کرم ضدآفتاب</b> و <b>عینک آفتابی</b> رتبه‌ی بالایی گرفته‌اند، چون کلمات‌شان («آفتاب»، «محافظت») به query شبیه است — اما سؤال ما درباره‌ی <b>لباس</b> بود، نه کرم یا عینک! این دقیقاً همان ضعف Bi-Encoder است.

</div>


## ۳. Cross-Encoder Reranker: مدل `bge-reranker-v2-m3`

<div dir="rtl">

مدل <a href="https://huggingface.co/BAAI/bge-reranker-v2-m3">BAAI/bge-reranker-v2-m3</a> یک Cross-Encoder چندزبانه با پشتیبانی خوب از فارسی است.
بار اول از HuggingFace دانلود می‌شود و بعد از آن لوکال اجرا می‌شود.

خروجی خام مدل <b>logit</b> است (می‌تواند منفی هم باشد).
برای تبدیل به بازه‌ی ۰ تا ۱ از تابع <b>sigmoid</b> استفاده می‌کنیم.

</div>


In [5]:
# pip install sentence-transformers langchain-classic

from sentence_transformers import CrossEncoder
import numpy as np

# لود مدل reranker (بار اول از HuggingFace دانلود می‌شود)
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)
print("Reranker loaded!")


Reranker loaded!


In [7]:
# ── مرحله ۲: Rerank با Cross-Encoder ──
# برای هر داکیومنت بازیابی‌شده، یک جفت (query, document) می‌سازیم
pairs = [(query, doc.page_content) for doc, _ in bi_results]

# مدل به هر جفت یک امتیاز می‌دهد
raw_scores = reranker.predict(pairs)

# تبدیل logit به احتمال بین ۰ و ۱
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

ce_scores = sigmoid(np.array(raw_scores))

# مرتب‌سازی مجدد بر اساس امتیاز cross-encoder
reranked = sorted(
    zip([doc for doc, _ in bi_results], ce_scores),
    key=lambda x: x[1],
    reverse=True
)

print(f"Cross-Encoder ranking for: '{query}'\n")
for i, (doc, score) in enumerate(reranked):
    print(f"  {i+1}. [{score*100:.1f}%] {doc.page_content[:70]}")


Cross-Encoder ranking for: 'چه لباسی پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت می‌کند؟'

  1. [53.4%] کرم ضدآفتاب SPF 50 پوست را در برابر آفتاب سوختگی محافظت می‌کند. مقاوم 
  2. [51.1%] پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های 
  3. [50.9%] کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طر
  4. [50.2%] چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی عالی از مچ پا در مسیرهای پیاده
  5. [50.1%] شلوار کوهنوردی سبک و سریع خشک‌شونده، مناسب مسیرهای طولانی در تابستان.
  6. [50.1%] عینک آفتابی پلاریزه با لنز UV400، چشم را در برابر نور شدید خورشید محاف


In [8]:
# ── مقایسه کنار هم: قبل و بعد از Rerank ──
print("-" * 95)
for i in range(len(bi_results)):
    before = bi_results[i][0].page_content[:40]
    after = reranked[i][0].page_content[:40]
    print(f"Before({i+1}): {before}")
    print(f"After({i+1}): {after}")
    print("-" * 95)


-----------------------------------------------------------------------------------------------
Before(1): پیراهن محافظ خورشید دارای استاندارد UPF 
After(1): کرم ضدآفتاب SPF 50 پوست را در برابر آفتا
-----------------------------------------------------------------------------------------------
Before(2): کرم ضدآفتاب SPF 50 پوست را در برابر آفتا
After(2): پیراهن محافظ خورشید دارای استاندارد UPF 
-----------------------------------------------------------------------------------------------
Before(3): چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی 
After(3): کلاه محافظ خورشید اشعه‌های فرابنفش را مس
-----------------------------------------------------------------------------------------------
Before(4): کلاه محافظ خورشید اشعه‌های فرابنفش را مس
After(4): چکمه‌های کوهنوردی ضدآب هستند و پشتیبانی 
-----------------------------------------------------------------------------------------------
Before(5): عینک آفتابی پلاریزه با لنز UV400، چشم را
After(5): شلوار کوهنوردی سبک و سریع خشک‌شونده، منا
-----

<div dir="rtl">
می‌بینید که Cross-Encoder داکیومنت‌های واقعاً مرتبط (پیراهن و کلاه محافظ خورشید) را به بالای لیست آورده و گمراه‌کننده‌ها (کرم، عینک) را پایین برده است — چون query و document را <b>با هم</b> خوانده و فهمیده که سؤال درباره‌ی «لباس» است.

</div>

## ۴. ادغام Reranker در LangChain

<div dir="rtl">
</div>
<div dir="rtl">
در LangChain، مفهوم reranking با <b>ContextualCompressionRetriever</b> پیاده‌سازی می‌شود:
</div>
<div dir="rtl">
یک retriever پایه (Bi-Encoder با k بزرگ) + یک compressor (همان reranker) که فقط بهترین‌ها را نگه می‌دارد.

</div>

```
base_retriever (k=10) → CrossEncoderReranker (top_n=3) → Docs نهایی
```


In [9]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# همان مدل reranker، این بار در قالب LangChain
ce_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

# compressor: از بین کاندیدها فقط ۳ تای برتر را نگه می‌دارد
compressor = CrossEncoderReranker(model=ce_model, top_n=3)

# retriever دو مرحله‌ای: اول k=10 با bi-encoder، بعد rerank
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)
print("Rerank retriever ready!")


Rerank retriever ready!


In [10]:
print(query)

چه لباسی پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت می‌کند؟


In [11]:
# تست retriever جدید
final_docs = rerank_retriever.invoke(query)

print(f"Top {len(final_docs)} docs after rerank:\n")
for i, doc in enumerate(final_docs):
    print(f"  {i+1}. {doc.page_content[:80]}")


Top 3 docs after rerank:

  1. کرم ضدآفتاب SPF 50 پوست را در برابر آفتاب سوختگی محافظت می‌کند. مقاوم در برابر آ
  2. پیراهن محافظ خورشید دارای استاندارد UPF 50+ است و ۹۸ درصد از اشعه‌های فرابنفش را
  3. کلاه محافظ خورشید اشعه‌های فرابنفش را مسدود می‌کند و لبه پهنی دارد. طراحی سبک.


## ۵. RAG کامل با Reranker

<div dir="rtl">

حالا دقیقاً مثل جلسات قبل یک RAG Chain با LCEL می‌سازیم — تنها تفاوت این است که به‌جای retriever ساده، از <code>rerank_retriever</code> استفاده می‌کنیم.

</div>


In [12]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "نمیدانم. اطلاعی در این رابطه ندارم"

Context:
{context}"""),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain با Rerank:  Retrieve(k=10) → Rerank(top 3) → Generate
rag_chain = (
    {"context": rerank_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# تست
response = rag_chain.invoke("چه لباسی پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت می‌کند؟")
print(response)


پیراهن محافظ خورشید با استاندارد UPF 50+ می‌تواند پوست بدن را هنگام پیاده‌روی در برابر آفتاب محافظت کند.


In [13]:
# تست 2
response = rag_chain.invoke("برای کوهنوردی در مسیر خیس و گل‌آلود چه کفشی پیشنهاد می‌کنید؟")
print(response)


چکمه‌های کوهنوردی ضدآب را پیشنهاد می‌کنم، زیرا این چکمه‌ها از پا در برابر رطوبت و گل محافظت می‌کنند و پشتیبانی عالی از مچ پا ارائه می‌دهند.


In [14]:
# تست 3 — سوال خارج از دانش
response = rag_chain.invoke("قیمت دلار امروز چند است؟")
print(response)


نمیدانم. اطلاعی در این رابطه ندارم


## ۶. مقایسه Bi-Encoder vs Cross-Encoder

| | Bi-Encoder (Embedding) | Cross-Encoder (Reranker) |
|---|---|---|
| **ورودی** | query و doc جداگانه encode می‌شوند | query و doc با هم به مدل داده می‌شوند |
| **خروجی** | بردار → cosine similarity | امتیاز ارتباط مستقیم |
| **سرعت** | بسیار سریع (بردارها از قبل محاسبه شده) | کند (هر جفت یک بار forward pass) |
| **دقت** | متوسط — تعامل کلمات را نمی‌فهمد | بالا — context کامل را می‌بیند |
| **مقیاس‌پذیری** | میلیون‌ها doc | فقط ده‌ها doc (کاندیدهای retrieve شده) |
| **نقش در Pipeline** | مرحله ۱: بازیابی گسترده (k=10~100) | مرحله ۲: مرتب‌سازی دقیق (top 3~5) |


## جمع‌بندی — نصب مورد نیاز

```bash
pip install langchain langchain-openai langchain-community langchain-classic faiss-cpu sentence-transformers
```

**Pipeline کامل RAG با Rerank:**
1. **Load / Split / Embed / Store** — مثل قبل
2. **Retrieve** — `vectorstore.as_retriever(search_kwargs={"k": 10})` (بازیابی گسترده)
3. **Rerank** — `CrossEncoderReranker` با مدل `BAAI/bge-reranker-v2-m3` (انتخاب top 3)
4. **Generate** — LCEL chain مثل قبل

<div dir="rtl">

<b>قانون سرانگشتی:</b> هر جا کیفیت retrieval مهم‌تر از سرعت است (پشتیبانی مشتری، اسناد حقوقی، پزشکی)،
reranker اضافه کنید. هزینه‌ی آن فقط کمی تأخیر است، اما دقت پاسخ نهایی به‌طور محسوسی بالا می‌رود.

</div>
